In [ ]:
#@title Download librerie
!pip install torch transformers accelerate datasets numpy tqdm scikit-learn wandb

In [ ]:
#@title Importazione moduli
!pip install -U datasets
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer

In [ ]:
#@title Importazione del modello da **Huggingface**
# model = AutoModelForMaskedLM.from_pretrained("cabrooks/LOGION-50k_wordpiece")
model = AutoModelForMaskedLM.from_pretrained('bowphs/PhilBerta')

model.num_parameters() / 1_000_000

In [ ]:
# @title Importazione del tokenizer del modello
# tokenizer = AutoTokenizer.from_pretrained("cabrooks/LOGION-50k_wordpiece")
tokenizer = AutoTokenizer.from_pretrained('bowphs/PhilBerta')
tokenizer.model_max_length = 512

In [ ]:
#@title Importazione del dataset
from datasets import load_dataset, DatasetDict

dataset = load_dataset("CNR-ILC/gs-maat-corpus")


In [ ]:
#@title Importazione del *data collator*
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [ ]:
#@title Tokenizzazione dei dati e raggruppamento in *chunk*
from itertools import chain

desired_columns = ['input_ids', 'attention_mask', 'labels'] 

def tokenize_function(examples):
    return tokenizer(examples["text"], padding=False, add_special_tokens=False)

chunk_size = 128
def group_texts(examples):
    # Prima si concatenano tutti i token
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    
    total_length = len(concatenated_examples["input_ids"])
    # Padding fino al prossimo chunk_size
    if total_length % chunk_size != 0:
        padding_length = chunk_size - (total_length % chunk_size)
        for key in concatenated_examples.keys():
            if key == "attention_mask":
                pad_value = 0
            else:
                pad_value = tokenizer.pad_token_id
            concatenated_examples[key] += [pad_value] * padding_length

    total_length = (len(concatenated_examples["input_ids"]) // chunk_size) * chunk_size
    # Spezza in chunk
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

def process_data(examples):
    tokenized_output = tokenize_function(examples)
    return group_texts(tokenized_output)

lm_datasets = dataset.map(process_data, batched=True, remove_columns=['text'])
lm_datasets = lm_datasets.select_columns(desired_columns)

In [ ]:
#@title TopK accuracy
import torch
import numpy as np
from sklearn.metrics import top_k_accuracy_score

def compute_metrics(eval_pred, compute_result=False):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    
    # Conversione sicura da torch (standard con accelerate) -> numpy (per calcolo top_k_accuracy_score) 
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()

    # Reshape di tutto per lavorare a livello dei singoli token
    logits = logits.reshape(-1, logits.shape[-1])    # (batch_size * sequence_length, vocab_size)
    labels = labels.flatten()                        # (batch_size * sequence_length)

    # Ora possiamo lavorare correttamente
    preds = np.argmax(logits, axis=1)
    
    # Solo per i token non mascherati (-100 è il valore ignorato nei label di Huggingface)
    mask = labels != -100
    if not np.any(mask):
        return {'top1_acc': 0,
        'top5_acc': 0,
        'top10_acc': 0,
        'top15_acc': 0,
        'top20_acc': 0,
        'top25_acc': 0,
        'top30_acc': 0,
        'top35_acc': 0,
        'top40_acc': 0,
        'top45_acc': 0,
        'top50_acc':0}
    
    preds = preds[mask]
    labels = labels[mask]
    logits = logits[mask]
    
    return {
        'top1_acc': (preds == labels).mean(),
        'top5_acc': top_k_accuracy_score(labels, logits, k=5, labels=np.arange(logits.shape[1])),
        'top10_acc': top_k_accuracy_score(labels, logits, k=10, labels=np.arange(logits.shape[1])),
        'top15_acc': top_k_accuracy_score(labels, logits, k=15, labels=np.arange(logits.shape[1])),
        'top20_acc': top_k_accuracy_score(labels, logits, k=20, labels=np.arange(logits.shape[1])),
        'top25_acc': top_k_accuracy_score(labels, logits, k=25, labels=np.arange(logits.shape[1])),
        'top30_acc': top_k_accuracy_score(labels, logits, k=30, labels=np.arange(logits.shape[1])),
        'top35_acc': top_k_accuracy_score(labels, logits, k=35, labels=np.arange(logits.shape[1])),
        'top40_acc': top_k_accuracy_score(labels, logits, k=40, labels=np.arange(logits.shape[1])),
        'top45_acc': top_k_accuracy_score(labels, logits, k=45, labels=np.arange(logits.shape[1])),
        'top50_acc': top_k_accuracy_score(labels, logits, k=50, labels=np.arange(logits.shape[1])),
    }

In [ ]:
#@title Definizione dei parametri di addestramento
import torch
from transformers import TrainingArguments
torch.cuda.empty_cache() #Pulisco la cache della GPU per liberare memoria prima del finetuning
training_args = TrainingArguments(
    output_dir="./gs-greBERTa",
    logging_dir="./greBERTa-logs", 
    report_to="wandb",         #weight & biases per il report delle metriche durante il finetuning
    eval_strategy="epoch",     # Valutazione basata sulle epoche
    save_strategy="no", 
    learning_rate=5e-5, # Passo     
    per_device_train_batch_size=16,
    per_device_eval_batch_size=1,
    num_train_epochs=10,             # Numero di epoche
    seed=42,                         # Seme per la riproducibilità
    fp16=True,                       # Precisione mista per migliorare le prestazioni
    logging_strategy="epoch",        # Log basati sulle epoche
    eval_accumulation_steps=1,       # Numero di batch che vengono elaborati sulla GPU prima di spostarli sulla CPU 
    batch_eval_metrics=True,         # Chiama `compute_metrics`, permettendo di non mantenere tutto in memoria 
    torch_empty_cache_steps=5000,     #Pulisce la cache ogni 5000 passi, evita problemi di OOM 
    dataloader_drop_last=False,       # Evita batch incompleti in validazione
)

In [ ]:
lm_datasets["test"][0]

In [ ]:
#@title finetuning (*continual-pretraining*) del modello
import math
import json 
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=lm_datasets["dev"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
) 

results = trainer.evaluate(eval_dataset=lm_datasets["test"])

#with open ("eval_results_Logion_normal.json", "w") as f: 
with open ("eval_results_PhilBERTA.json", "w") as f: 
     json.dump(results, f, indent=4)
#eval_results = trainer.evaluate(eval_dataset=lm_datasets["test"])
#print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")